# Overmerge Cluster Analysis

Process one author at a time: query embeddings, compute metrics, print progress.

- Approach 2: K=2 silhouette on work embeddings
- Approach 3: Pairwise work-to-work cosine similarity
- Approach 4: Per-work fit scores
- Job: #105.11 aer-overmerge-signal-calibration

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
import time
from datetime import datetime
from pyspark.sql.types import StructType, StructField, LongType, StringType, IntegerType, DoubleType

WORK_EMBEDDINGS_TABLE = "openalex.vector_search.work_embeddings_v2"
WORKS_TABLE = "openalex.works.openalex_works"
EMBEDDING_DIM = 1024
MIN_WORKS = 5
CHECKPOINT_TABLE = "openalex.aer.overmerge_cluster_analysis"

# Schema for checkpoint table
SCHEMA = StructType([
    StructField("author_id", LongType()),
    StructField("label", StringType()),
    StructField("work_count", IntegerType()),
    StructField("silhouette_k2", DoubleType()),
    StructField("mean_pairwise", DoubleType()),
    StructField("min_pairwise", DoubleType()),
    StructField("p10_pairwise", DoubleType()),
    StructField("std_pairwise", DoubleType()),
    StructField("min_fit", DoubleType()),
    StructField("p5_fit", DoubleType()),
    StructField("p10_fit", DoubleType()),
    StructField("mean_fit", DoubleType()),
    StructField("std_fit", DoubleType()),
    StructField("frac_below_05", DoubleType()),
])

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CHECKPOINT_TABLE} (
        author_id BIGINT, label STRING, work_count INT,
        silhouette_k2 DOUBLE,
        mean_pairwise DOUBLE, min_pairwise DOUBLE, p10_pairwise DOUBLE, std_pairwise DOUBLE,
        min_fit DOUBLE, p5_fit DOUBLE, p10_fit DOUBLE, mean_fit DOUBLE, std_fit DOUBLE,
        frac_below_05 DOUBLE
    )
""")

done_ids = set(r['author_id'] for r in spark.sql(f"SELECT author_id FROM {CHECKPOINT_TABLE}").collect())
print(f"Checkpoint table has {len(done_ids)} authors already computed")

## Define labeled authors

In [ ]:
# Zendesk overmerge cases
zendesk_overmerged = [
    5106038276, 5127355048, 5121306932, 5071424520, 5088402891,
    5045317380, 5100726594, 5108072493, 5009709588, 5045904720,
    5012309101, 5069151214, 5011165427
]

# Gold standard splits (overmerged)
gold_splits = [
    5085619921, 5041864386, 5117075932, 5110196385, 5065096990,
    5037835036, 5071539405, 5017142455, 5110027945, 5091859986,
    5108623528, 5101092520, 5070168570, 5072305232, 5013478230,
    5009307852, 5016882470, 5091442005, 5079979698, 5108570566,
    5110358878, 5079166689, 5108748587, 5102755488, 5079652777,
    5113385911, 5113952815, 5010422511, 5007114677, 5037924045,
    5025452752, 5088152507, 5073218194, 5112257425, 5061138371,
    5033771456, 5101839347, 5074059264, 5080312909, 5010975613,
    5048151405, 5052571385, 5111372795, 5112390495, 5103459170,
    5103347958, 5062125091, 5110749239, 5021173676, 5113857253,
    5108453412, 5076836352, 5060512575, 5079413276, 5079617160,
    5055163904, 5064679635, 5109293548, 5067290858, 5050939922,
    5056229359, 5037435179, 5006626689, 5103976114, 5010484703,
    5087117526, 5019217807, 5075755572, 5109148503, 5014188192,
    5051189473, 5058421597, 5057408903, 5056945568, 5074432852,
    5019862321, 5103907138, 5003683328, 5103173829, 5051468011,
    5038742031, 5062220638, 5114098473, 5001106162, 5029904482,
    5073480387, 5002981530
]

# Gold standard confirmed (clean)
gold_confirmed = [
    5087166684, 5081747147, 5082325782, 5012168214, 5012434282,
    5003686344, 5010617958, 5084195402, 5079654941, 5000848656,
    5061377970, 5000447750, 5023844756, 5080591937, 5027129242,
    5067354901, 5109968665, 5037064192, 5072647734, 5016361065,
    5026928687, 5002615359, 5027008194, 5063093897, 5108260432,
    5011946452, 5079497159, 5083508133, 5109604058, 5079798553,
    5088931673, 5060557374, 5086191294, 5044638100, 5036926833,
    5037676122, 5082406391, 5073380969, 5019534047, 5090821911,
    5111962545, 5027442102, 5036629892, 5084506603, 5075560544,
    5114030225, 5047130741, 5088150711, 5024736221, 5046646750,
    5111945307, 5066075206, 5063684507, 5053820618, 5055178909,
    5087090962, 5103243948, 5050549195, 5033358050, 5102850112,
    5055945041, 5026328537, 5022112939, 5011576899, 5036961392,
    5000185973, 5013710631, 5088159296, 5063641432, 5018449545,
    5074749021, 5052625144, 5078623160, 5030659236, 5004860476,
    5002085848, 5088382884, 5050861768, 5070669440, 5035638193,
    5027340715, 5067395854, 5018332726, 5000163809, 5048092761,
    5063197873, 5077040304, 5033702363, 5110246466, 5074731779,
    5015418171, 5052638321, 5085326237, 5061312215, 5075570397,
    5112171421, 5050680576, 5017797591, 5065396267, 5087110857
]

overmerged_set = set(zendesk_overmerged + gold_splits)
clean_set = set(gold_confirmed)
all_authors = list(overmerged_set | clean_set)

print(f"Overmerged: {len(overmerged_set)}, Clean: {len(clean_set)}, Total: {len(all_authors)}")

## Helper functions

In [ ]:
def get_author_embeddings(author_id):
    """Pull work embeddings for a single author. Returns list of 1024-dim arrays."""
    rows = spark.sql(f"""
        SELECT e.work_id, e.embedding
        FROM {WORK_EMBEDDINGS_TABLE} e
        JOIN {WORKS_TABLE} w ON e.work_id = CAST(w.id AS STRING)
        LATERAL VIEW OUTER EXPLODE(w.authorships) AS a
        WHERE a.author.id = 'https://openalex.org/A{author_id}'
          AND e.embedding IS NOT NULL
    """).collect()
    embeddings = []
    work_ids = []
    for row in rows:
        emb = np.array(row['embedding'][:EMBEDDING_DIM], dtype=np.float64)
        if not np.any(np.isnan(emb)):
            embeddings.append(emb)
            work_ids.append(row['work_id'])
    return embeddings, work_ids


def compute_silhouette_k2(embeddings):
    """K-means k=2, return silhouette score."""
    if len(embeddings) < 4:
        return None
    X = np.array(embeddings)
    km = KMeans(n_clusters=2, n_init=10, random_state=42)
    labels = km.fit_predict(X)
    if len(set(labels)) < 2:
        return None
    return silhouette_score(X, labels, metric='cosine')


def compute_pairwise_stats(embeddings):
    """Pairwise cosine similarity summary stats."""
    X = np.array(embeddings)
    sim_matrix = cos_sim(X)
    n = len(X)
    upper_tri = sim_matrix[np.triu_indices(n, k=1)]
    return {
        'mean_pairwise': float(np.mean(upper_tri)),
        'min_pairwise': float(np.min(upper_tri)),
        'p10_pairwise': float(np.percentile(upper_tri, 10)),
        'std_pairwise': float(np.std(upper_tri)),
    }


def compute_per_work_fit(embeddings):
    """Per-work mean cosine similarity to all other works."""
    X = np.array(embeddings)
    sim_matrix = cos_sim(X)
    n = len(X)
    fit_scores = []
    for i in range(n):
        others = [sim_matrix[i][j] for j in range(n) if j != i]
        fit_scores.append(float(np.mean(others)))
    return fit_scores


print("Helper functions defined.")

## Process authors one at a time

In [ ]:
todo = [a for a in all_authors if a not in done_ids]
total = len(todo)
already = len(done_ids)
errors = []
start_time = time.time()

print(f"Processing {total} authors ({already} already checkpointed, {total + already} total)")
print("=" * 90)

for i, author_id in enumerate(todo):
    author_start = time.time()
    label = 'overmerged' if author_id in overmerged_set else 'clean'

    try:
        embeddings, work_ids = get_author_embeddings(author_id)
        n_works = len(embeddings)

        if n_works < MIN_WORKS:
            elapsed = time.time() - start_time
            rate = (i + 1) / elapsed if elapsed > 0 else 0
            eta = (total - i - 1) / rate if rate > 0 else 0
            print(f"  [{i+1}/{total}] A{author_id} ({label}): {n_works} works < {MIN_WORKS}, skipped | {elapsed:.0f}s elapsed, ETA {eta:.0f}s")
            continue

        # Compute all metrics
        sil = compute_silhouette_k2(embeddings)
        pw = compute_pairwise_stats(embeddings)
        fit_scores = compute_per_work_fit(embeddings)

        row = (
            author_id, label, n_works, sil,
            pw['mean_pairwise'], pw['min_pairwise'], pw['p10_pairwise'], pw['std_pairwise'],
            float(np.min(fit_scores)), float(np.percentile(fit_scores, 5)),
            float(np.percentile(fit_scores, 10)), float(np.mean(fit_scores)),
            float(np.std(fit_scores)),
            float(sum(1 for s in fit_scores if s < 0.5) / len(fit_scores)),
        )

        # Checkpoint: write to Delta immediately
        spark.createDataFrame([row], schema=SCHEMA).write.mode("append").saveAsTable(CHECKPOINT_TABLE)

        author_elapsed = time.time() - author_start
        elapsed = time.time() - start_time
        rate = (i + 1) / elapsed if elapsed > 0 else 0
        eta = (total - i - 1) / rate if rate > 0 else 0
        print(
            f"  [{i+1}/{total}] A{author_id} ({label}, {n_works} works): "
            f"p10_pw={pw['p10_pairwise']:.3f} min_fit={float(np.min(fit_scores)):.3f} "
            f"| {author_elapsed:.1f}s | {elapsed:.0f}s elapsed, ETA {eta:.0f}s"
        )

    except Exception as e:
        errors.append({'author_id': author_id, 'error': str(e)})
        elapsed = time.time() - start_time
        print(f"  [{i+1}/{total}] A{author_id} ERROR: {e} | {elapsed:.0f}s elapsed")

total_elapsed = time.time() - start_time
final_count = spark.sql(f"SELECT COUNT(*) AS n FROM {CHECKPOINT_TABLE}").collect()[0]['n']
print("=" * 90)
print(f"Done: {final_count} authors in checkpoint table, {len(errors)} errors, {total_elapsed:.0f}s total")

## Summary

In [ ]:
# Read all results from checkpoint table
all_results = [r.asDict() for r in spark.sql(f"SELECT * FROM {CHECKPOINT_TABLE}").collect()]
print(f"Loaded {len(all_results)} authors from checkpoint table")

output_lines = []
output_lines.append("ALL METRICS - separation (median overmerged vs median clean):")
output_lines.append("=" * 80)

all_metrics = [
    'silhouette_k2',
    'mean_pairwise', 'min_pairwise', 'p10_pairwise', 'std_pairwise',
    'min_fit', 'p5_fit', 'p10_fit', 'std_fit', 'frac_below_05',
]

for metric in all_metrics:
    over_vals = [r[metric] for r in all_results if r['label'] == 'overmerged' and r.get(metric) is not None]
    clean_vals = [r[metric] for r in all_results if r['label'] == 'clean' and r.get(metric) is not None]
    if over_vals and clean_vals:
        over_med = np.median(over_vals)
        clean_med = np.median(clean_vals)
        gap = abs(over_med - clean_med)
        pooled_std = np.std(over_vals + clean_vals)
        effect_size = gap / pooled_std if pooled_std > 0 else 0
        direction = 'higher' if over_med > clean_med else 'lower'
        line = f"  {metric:20s}: overmerged={over_med:.4f} clean={clean_med:.4f} gap={gap:.4f} effect={effect_size:.2f} ({direction})"
        output_lines.append(line)

output_lines.append("")
output_lines.append("APPROACH 4 - Per-Work Fit Score:")
for label in ['overmerged', 'clean']:
    subset = [r for r in all_results if r['label'] == label]
    output_lines.append(f"  {label} (n={len(subset)}):")
    for metric in ['min_fit', 'p5_fit', 'p10_fit', 'mean_fit', 'std_fit', 'frac_below_05']:
        vals = [r[metric] for r in subset if r.get(metric) is not None]
        if vals:
            output_lines.append(f"    {metric}: mean={np.mean(vals):.4f} median={np.median(vals):.4f} q1={np.percentile(vals,25):.4f} q3={np.percentile(vals,75):.4f}")

output = "\n".join(output_lines)
print(output)
dbutils.notebook.exit(output)